# Phase 15: Master Pipeline Test
This notebook verifies the end-to-end functionality of the Mini NotebookLM pipeline.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from src.master_pipeline import MasterPipeline
import shutil
import csv
print("Imports successful!")

## 1. Setup and Ingestion

In [ ]:
# Cleanup past tests
for p in ["test_master_faiss", "test_master.db", "test_master_graph"]:
    if os.path.exists(p):
        if os.path.isdir(p): shutil.rmtree(p)
        else: os.remove(p)

pipeline = MasterPipeline()

# Create dummy CSV for ingestion
csv_path = "test_dummy.csv"
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["Concept", "Definition"])
    writer.writerow(["Machine Learning", "A subfield of AI that focuses on building systems that learn from data."])
    writer.writerow(["Neural Networks", "Computing systems inspired by the biological neural networks that constitute animal brains."])

source_id = pipeline.ingest(file_path=csv_path, source_type="csv")
print(f"Ingested CSV with Source ID: {source_id}")
assert source_id is not None

## 2. Test Generation and Modes

In [ ]:
# Configure mock LLM client to avoid API calls during test
class MockLLM:
    def invoke(self, prompt):
        return "This is a generated response based on: Neural Networks. [SOURCE_1]"
    def stream(self, prompt):
        yield "This "
        yield "is a "
        yield "stream."

pipeline.set_llm(provider="mock", model="mock", api_key="mock")
pipeline.llm = MockLLM() # Override with our mock

# 1. Chat Mode
pipeline.set_mode("chat")
response1 = pipeline.generate("Tell me about Neural Networks")
print(f"Chat Response: {response1}")
assert "Neural Networks" in response1

# 2. Study Mode
pipeline.set_mode("study")
response2 = pipeline.generate("How do Neural Networks relate to AI?")
print(f"Study Response: {response2}")

# Check stats
stats = pipeline.get_stats()
print(f"Pipeline Stats: {stats}")

# Cleanup
if os.path.exists(csv_path): os.remove(csv_path)
print("Master Pipeline tests passed!")